# 02 — SDXL transfer pipeline

This repeats the key SD1.5 checks on a smaller matched subset.

In [ ]:
from pathlib import Path
import os, sys

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT / "src"))
os.environ["PYTHONPATH"] = str(PROJECT_ROOT / "src") + os.pathsep + os.environ.get("PYTHONPATH", "")
print("PROJECT_ROOT =", PROJECT_ROOT)

from diffusion_attention_analysis_v2.notebook_runner import run_config, run_suite, show_report

In [ ]:
SMOKE_TEST = True
CAPTURE_OVERRIDES = ["data.max_prompts=4", "runtime.num_seeds=1"] if SMOKE_TEST else []
SAE_OVERRIDES = ["sae.num_epochs=1", "sae.max_activation_files=16", "sae.max_tokens_per_file=1024"] if SMOKE_TEST else []
INTERVENTION_OVERRIDES = ["data.max_prompts=2", "runtime.num_seeds=1"] if SMOKE_TEST else []

## Stage 1 — capture

In [ ]:
run_config("configs/01_capture/sdxl_transfer.yaml", overrides=CAPTURE_OVERRIDES)

## Stage 2 — attention localization

In [ ]:
run_config("configs/02_attention_localization/sdxl_transfer.yaml")
show_report("outputs/sdxl/02_attention_localization/report.json")

## Stage 3 — SAE training

In [ ]:
run_config("configs/03_sae_training/sdxl_transfer.yaml", overrides=SAE_OVERRIDES)
show_report("outputs/sdxl/03_sae_mid/report.json")

## Stage 4 — concept dictionary

In [ ]:
run_config("configs/04_concept_dictionary/sdxl_transfer.yaml", overrides=(["dictionary.max_activation_files=16"] if SMOKE_TEST else []))
show_report("outputs/sdxl/04_concept_dictionary/report.json")

## Stage 5 — transfer interventions

In [ ]:
run_config("configs/05_interventions/sdxl_map_reweight.yaml", overrides=INTERVENTION_OVERRIDES)
show_report("outputs/sdxl/05_intervention_map_reweight/report.json")

In [ ]:
run_config("configs/05_interventions/sdxl_sae_steering.yaml", overrides=INTERVENTION_OVERRIDES)
show_report("outputs/sdxl/05_intervention_sae_steering/report.json")